# Stochastic Calculus, Step by Step · Lesson 5: Lognormal prices and Jensen's inequality

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The lognormal share · 3. Mean, median, mode and variance · 4. Jensen's inequality and the σ²/2 · 5. The central limit theorem · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 5,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. The lognormal share

If the log return is normal, $\ln(S_T/S_0) = (r - \tfrac12\sigma^2)T + \sigma\sqrt T Z$, the price $S_T$ is **lognormal**: always positive, skewed to the right.

In [ ]:
rng = np.random.default_rng(seed)
Z = rng.standard_normal(2_000_000)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)

## 3. Mean, median, mode and variance

From $E[e^{uZ}] = e^{u^2/2}$ (Lesson 4): $E[S_T] = S_0e^{rT}$ and $E[S_T^2] = S_0^2e^{(2r + \sigma^2)T}$, so $\mathrm{Var}(S_T) = S_0^2e^{2rT}(e^{\sigma^2T} - 1)$. The median is $S_0e^{(r - \frac12\sigma^2)T}$ and the mode $S_0e^{(r - \frac32\sigma^2)T}$.

In [ ]:
mean = S0 * math.exp(r * T)
var = S0**2 * math.exp(2 * r * T) * (math.exp(sigma**2 * T) - 1)
ln = {"mean": mean, "mc_mean": float(ST.mean()), "second": S0**2 * math.exp((2 * r + sigma**2) * T),
      "var": var, "sd": math.sqrt(var), "mc_sd": float(ST.std()),
      "median": S0 * math.exp((r - 0.5 * sigma**2) * T), "mc_median": float(np.median(ST)),
      "mode": S0 * math.exp((r - 1.5 * sigma**2) * T), "share_above_mean": float((ST > mean).mean())}
assert abs(ln["mc_mean"] / mean - 1) < 1e-3 and abs(ln["mc_sd"] / ln["sd"] - 1) < 3e-3
out["ln"] = ln
pd.Series(ln)

## 4. Jensen's inequality and the σ²/2

For a convex $f$, $E[f(X)] \ge f(E[X])$. With $f = \exp$: $E[e^{\sigma Z}] = e^{\sigma^2/2} > e^0 = 1$. So if we forgot the $-\tfrac12\sigma^2$ and wrote $S_T = S_0e^{rT + \sigma\sqrt TZ}$, the average would be too big by the factor $e^{\sigma^2T/2}$: the $-\tfrac12\sigma^2$ is exactly what keeps the mean at $S_0e^{rT}$. The call payoff is convex too: $E[\max(S_T - K, 0)] > \max(E[S_T] - K, 0)$.

In [ ]:
naive = S0 * np.exp(r * T + sigma * math.sqrt(T) * Z)
call_avg = float(np.maximum(ST - K, 0).mean())
out["jensen"] = {"naive_mean": float(naive.mean()), "naive_theory": mean * math.exp(sigma**2 * T / 2),
                 "factor": math.exp(sigma**2 * T / 2), "call_avg": call_avg, "call_at_mean": max(mean - K, 0.0),
                 "e_log": float(np.log(ST / S0).mean()), "log_e": math.log(mean / S0),
                 "drift_L": (r - 0.5 * sigma**2) * T, "half_var": 0.5 * sigma**2 * T,
                 "median_over_mean": math.exp(-0.5 * sigma**2 * T)}
out["jensen"]

## 5. The central limit theorem

Why normal log returns? A year's log return is the sum of many small, independent daily ones. Even if each day is just a coin flip, up or down by $\sigma\sqrt{\Delta t}$, the sum is close to normal, and the year-end price close to lognormal.

In [ ]:
days, n_paths = 252, 400_000
dt = T / days
ups = rng.binomial(days, 0.5, n_paths)                      # number of up days
logret = days * (r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * (2 * ups - days)
SB = S0 * np.exp(logret)
# Each possible number of up days k gives one price; divide its frequency by the gap between neighbouring prices
k = np.arange(days + 1)
lattice = S0 * np.exp(days * (r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * (2 * k - days))
freq = np.bincount(ups, minlength=days + 1) / n_paths
gap = np.gradient(lattice)
keep = (lattice > 40) & (lattice < 220)
s = np.linspace(40, 220, 361)
dens = np.exp(-(np.log(s / S0) - (r - 0.5 * sigma**2) * T) ** 2 / (2 * sigma**2 * T)) / (s * sigma * math.sqrt(2 * math.pi * T))
qs = [0.01, 0.1, 0.5, 0.9, 0.99]
qz = {0.01: -2.3263478740, 0.1: -1.2815515655, 0.5: 0.0, 0.9: 1.2815515655, 0.99: 2.3263478740}
out["clt"] = {"days": days, "paths": n_paths, "mean": float(SB.mean()), "sd": float(SB.std()),
              "quantiles": [{"q": q, "coin": float(np.quantile(SB, q)),
                             "lognormal": S0 * math.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * qz[q])} for q in qs],
              "fig": {"mid": lattice[keep].round(3).tolist(), "hist": (freq / gap)[keep].round(6).tolist(),
                      "s": s.round(3).tolist(), "dens": dens.round(6).tolist(), "mean": mean, "median": ln["median"], "mode": ln["mode"],
                      "ticks": [{"x": v, "label": f"${v}"} for v in range(40, 221, 20)]}}
pd.DataFrame(out["clt"]["quantiles"])

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** Use $E[e^{uZ}] = e^{u^2/2}$ to derive $E[S_T]$, $E[S_T^2]$ and $\mathrm{Var}(S_T)$ for the lognormal share. Evaluate the standard deviation with the course's numbers.

**Exercise 2.** Show that the median of $S_T$ is $S_0e^{(r - \frac12\sigma^2)T}$. Why is it below the mean?

**Exercise 3.** Apply Jensen's inequality to $f(x) = x^2$. What familiar fact do you get?

**Exercise 4.** Find the mode of $S_T$ (the peak of its density) by differentiating the lognormal density, and evaluate it.

**Exercise 5.** The put payoff $\max(K - s, 0)$ is convex. Use Jensen's inequality to give a lower bound for $E[\max(K - S_T, 0)]$. Then compute the exact value, using $\max(K - s, 0) = \max(s - K, 0) - s + K$ and Lesson 4's formula for the call.

**Exercise 6.** A year's log return is the sum of 252 independent daily log returns, each with mean $0.01/252$ and standard deviation $0.2/\sqrt{252}$, but with an unknown distribution. Use the central limit theorem to approximate the probability that the year's log return exceeds 0.2.

**Exercise 7.** (Code) Simulate $S_T$ with and without the $-\tfrac12\sigma^2$, for $\sigma = 20\%$ and $\sigma = 40\%$. Compare each average with $S_0e^{rT}$.

**Exercise 8.** (Code) Replace the coin-flip days of Section 5 with days whose log returns are uniform with the same mean and variance. Compare the 1% and 99% quantiles of $S_T$ with the lognormal ones.

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")